# Guardian Football — Function Calling Agent

Loads the annotated Guardian football data and exposes 5 tools that Azure OpenAI can call to answer free-text questions about players and teams.

**Prerequisite:** run `GuardianFootballAnnotation.ipynb` first to generate `guardian_football_annotated.pkl`.

In [44]:
## Cell 1 — Setup: load data and initialise Azure OpenAI client

import pickle
import json
from urllib.parse import urlparse
from dotenv import find_dotenv
from openai import AzureOpenAI

# ── parse .env ───────────────────────────────────────────────────────────────
def read_env(path):
    env = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            env[key.strip()] = val.strip().strip('"').strip("'")
    return env

env = read_env(find_dotenv())

raw_base       = env.get("GPT_BASE", "")
parsed         = urlparse(raw_base)
azure_endpoint = f"{parsed.scheme}://{parsed.netloc}"
api_key        = env.get("GPT_KEY", "")
api_version    = env.get("GPT_VERSION", "2025-04-01-preview")
CHAT_MODEL     = env.get("GPT_CHAT_MODEL", "gpt-4o-mini-context-course-2026-afthibara")

client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=api_key,
    api_version=api_version,
)

# ── load annotated data ───────────────────────────────────────────────────────
with open("guardian_football_annotated_all.pkl", "rb") as f:
    enriched_articles = pickle.load(f)

all_players = sorted({p for art in enriched_articles for p in art["players"]})
all_teams   = sorted({t for art in enriched_articles for t in art["teams"]})

print(f"Loaded {len(enriched_articles)} annotated articles.")
print(f"Players in data ({len(all_players)}): {', '.join(all_players[:10])}{'…' if len(all_players) > 10 else ''}")
print(f"Teams   in data ({len(all_teams)}):   {', '.join(all_teams[:10])}{'…' if len(all_teams) > 10 else ''}")


Loaded 50 annotated articles.
Players in data (519):  Martínez, A Gomes, Abdukodir Khusanov, Abraham, Adam Armstrong, Adam Forshaw, Adams, Adarabioyo, Ademola Lookman, Adli…
Teams   in data (175):   1. FC Nürnberg II, AFC Bournemouth, AFC Wimbledon, Accrington Stanley, Arsenal, Aston Villa, Atalanta, Athletic Bilbao, Atlético Madrid, Barcelona…


In [ ]:
## Cell 2 — The 5 tool functions

import re

MAX_PARAGRAPHS = 5  # cap context sent to GPT to stay within token limits
_verbose = False    # set True by dispatcher when show_messages=True

def _lookup_paragraphs(name: str) -> list[dict]:
    """Return up to MAX_PARAGRAPHS dicts with 'headline' and 'text' for paragraphs mentioning name."""
    name_lower = name.lower()
    results = []
    for art in enriched_articles:
        for para in art["paragraphs"]:
            if name_lower in para["text"].lower():
                results.append({"headline": art["headline"], "text": para["text"]})
                if len(results) >= MAX_PARAGRAPHS:
                    return results
    return results

def _build_context(paragraphs: list[dict]) -> str:
    """Format paragraphs into structured context entries."""
    entries = []
    for p in paragraphs:
        entries.append(
            f"In the article titled: {p['headline']}\n"
            f"It was written that: {p['text']}"
        )
    return "\n\n".join(entries)

def _gpt(system: str, user: str) -> str:
    """Single GPT call helper."""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
    )
    return resp.choices[0].message.content.strip()

def _enforce_sentences(text: str, n: int) -> str:
    """Hard fallback: keep only the first n sentences if the model over-generates."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return " ".join(sentences[:n])

def _print_context(label: str, context: str):
    """Print the retrieved context when verbose mode is on."""
    print(f"\n── CONTEXT: {label} {'─'*(50 - len(label))}")
    for i, entry in enumerate(context.split("\n\n"), 1):
        print(f"  [{i}] {entry[:200]}{'…' if len(entry) > 200 else ''}")

# Shared system prompts
_SUMMARY_SYSTEM = (
    "You are a football assistant. Your replies are always EXACTLY two sentences — never more, never less. "
    "No preamble, no bullet points, no headers. "
    "Speak naturally without mentioning sources, articles, or the Guardian. "
    "Use only the facts provided. Stop after sentence two."
)

_ANSWER_SYSTEM = (
    "You are a football assistant. Your replies are always EXACTLY one sentence — never more, never less. "
    "No preamble, no bullet points, no headers. "
    "Speak naturally without mentioning sources, articles, or the Guardian. "
    "Use only the facts provided. Stop after sentence one."
)

# ── 1. How many articles mention a player or team ───────────────────────────
def how_many_articles(name: str) -> str:
    """Count articles that mention the given player or team name."""
    name_lower = name.lower()
    count = sum(
        1 for art in enriched_articles
        if any(p.lower() == name_lower for p in art["players"])
        or any(t.lower() == name_lower for t in art["teams"])
    )
    total = len(enriched_articles)
    return f"Out of the last {total} articles in the Guardian, {count} have mentioned {name}."

# ── 2. Two-sentence summary about a team ────────────────────────────────────
def tell_me_about_team(team: str) -> str:
    paras = _lookup_paragraphs(team)
    if not paras:
        return f"There's no recent news about {team}."
    context = _build_context(paras)
    if _verbose:
        _print_context(f"tell_me_about_team({team})", context)
    raw = _gpt(
        _SUMMARY_SYSTEM,
        f"{context}\n\nSummarise the latest news about {team}. Two sentences only. Stop after sentence two."
    )
    return _enforce_sentences(raw, 2)

# ── 3. Two-sentence summary about a player ──────────────────────────────────
def tell_me_about_player(player: str) -> str:
    paras = _lookup_paragraphs(player)
    if not paras:
        return f"There's no recent news about {player}."
    context = _build_context(paras)
    if _verbose:
        _print_context(f"tell_me_about_player({player})", context)
    raw = _gpt(
        _SUMMARY_SYSTEM,
        f"{context}\n\nSummarise the latest news about {player}. Two sentences only. Stop after sentence two."
    )
    return _enforce_sentences(raw, 2)

# ── 4. Answer a specific question about a team ──────────────────────────────
def answer_team_question(team: str, question: str) -> str:
    paras = _lookup_paragraphs(team)
    if not paras:
        return f"There's no recent news about {team}."
    context = _build_context(paras)
    if _verbose:
        _print_context(f"answer_team_question({team})", context)
    raw = _gpt(
        _ANSWER_SYSTEM,
        f"{context}\n\nQuestion: {question}\nOne sentence only. Stop after sentence one."
    )
    return _enforce_sentences(raw, 1)

# ── 5. Answer a specific question about a player ────────────────────────────
def answer_player_question(player: str, question: str) -> str:
    paras = _lookup_paragraphs(player)
    if not paras:
        return f"There's no recent news about {player}."
    context = _build_context(paras)
    if _verbose:
        _print_context(f"answer_player_question({player})", context)
    raw = _gpt(
        _ANSWER_SYSTEM,
        f"{context}\n\nQuestion: {question}\nOne sentence only. Stop after sentence one."
    )
    return _enforce_sentences(raw, 1)

print("Functions defined: how_many_articles, tell_me_about_team, tell_me_about_player, "
      "answer_team_question, answer_player_question")


Functions defined: how_many_articles, tell_me_about_team, tell_me_about_player, answer_team_question, answer_player_question


In [46]:
## Cell 3 — OpenAI tool specs (JSON schema)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "how_many_articles",
            "description": (
                "Returns the number of articles in the dataset that mention a given "
                "football player or team. Use this when the user asks how much someone "
                "has been in the news, how often a team is covered, or how many times "
                "a name appears."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The player or team name to look up."
                    }
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tell_me_about_team",
            "description": (
                "Retrieves recent news paragraphs about a football club or national team "
                "and returns a two-sentence summary. Use this when the user writes just a "
                "team name, asks for a summary of a team, or asks what is new with a team."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "team": {
                        "type": "string",
                        "description": "The football club or national team name."
                    }
                },
                "required": ["team"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tell_me_about_player",
            "description": (
                "Retrieves recent news paragraphs about a football player and returns a "
                "two-sentence summary. Use this when the user writes just a player name, "
                "asks for a summary of a player, or asks what is new with a player."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "player": {
                        "type": "string",
                        "description": "The football player's name."
                    }
                },
                "required": ["player"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "answer_team_question",
            "description": (
                "Finds recent news paragraphs about a football team and answers a specific "
                "question about them in one sentence. Use this when the user asks a specific "
                "question about a team (e.g. 'Who scored for Arsenal?', 'Did Chelsea win?')."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "team": {
                        "type": "string",
                        "description": "The football club or national team name."
                    },
                    "question": {
                        "type": "string",
                        "description": "The specific question to answer about the team."
                    }
                },
                "required": ["team", "question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "answer_player_question",
            "description": (
                "Finds recent news paragraphs about a football player and answers a specific "
                "question about them in one sentence. Use this when the user asks a specific "
                "question about a player (e.g. 'Did Haaland score?', 'Is Salah injured?')."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "player": {
                        "type": "string",
                        "description": "The football player's name."
                    },
                    "question": {
                        "type": "string",
                        "description": "The specific question to answer about the player."
                    }
                },
                "required": ["player", "question"]
            }
        }
    },
]

print(f"Defined {len(TOOLS)} tools: {[t['function']['name'] for t in TOOLS]}")


Defined 5 tools: ['how_many_articles', 'tell_me_about_team', 'tell_me_about_player', 'answer_team_question', 'answer_player_question']


In [47]:
## Cell 4 — Dispatcher: routes a question to one or more tools and returns the answer

import textwrap

# Map tool names → Python callables
FUNCTION_MAP = {
    "how_many_articles":      lambda args: how_many_articles(args["name"]),
    "tell_me_about_team":     lambda args: tell_me_about_team(args["team"]),
    "tell_me_about_player":   lambda args: tell_me_about_player(args["player"]),
    "answer_team_question":   lambda args: answer_team_question(args["team"], args["question"]),
    "answer_player_question": lambda args: answer_player_question(args["player"], args["question"]),
}

_DISPATCH_SYSTEM = (
    "You are a football news assistant with access to lookup tools. "
    "Rules for tool selection:\n"
    "1. If the user asks a specific question about a TEAM (e.g. 'What problems is Arsenal facing?', "
    "'Did Chelsea win?'), call BOTH tell_me_about_team AND answer_team_question.\n"
    "2. If the user asks a specific question about a PLAYER (e.g. 'Did Haaland score?', "
    "'Is Salah injured?'), call BOTH tell_me_about_player AND answer_player_question.\n"
    "3. If the user just asks for a summary or mentions a name only, call tell_me_about_team "
    "or tell_me_about_player.\n"
    "4. If the user asks how often someone appears in the news, call how_many_articles.\n"
    "Always call tools — never answer from your own knowledge."
)

_COMBINE_SYSTEM = (
    "You are a football assistant giving a conversational reply. "
    "You will be given a user question and one or more tool results. "
    "Write a single fluent reply that keeps all the information from the results "
    "but removes any repetition. Do not mention sources, articles, or the Guardian. "
    "Match the length of the combined results — do not expand or add new facts."
)

def _print_message(msg, label=None):
    """Pretty-print a single message dict or ChatCompletionMessage object."""
    if hasattr(msg, "role"):
        role = msg.role
        content = msg.content or ""
        tool_calls = getattr(msg, "tool_calls", None)
    else:
        role = msg.get("role", "?")
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls")

    prefix = f"[{label or role.upper()}]"
    print(f"\n{'─'*60}")
    print(f"{prefix}")
    if content:
        print(textwrap.fill(content, width=80, initial_indent="  ", subsequent_indent="  "))
    if tool_calls:
        for tc in tool_calls:
            if hasattr(tc, "function"):
                print(f"  tool_call → {tc.function.name}({tc.function.arguments})")


def dispatch(question: str, show_messages: bool = False) -> str:
    """
    1. Ask GPT which tool(s) to call.
    2. Execute each tool (printing retrieved context when show_messages=True).
    3. If more than one result, use a second GPT call to merge them into a
       single fluent reply without repetition.
       If only one result, return it directly.
    """
    global _verbose
    _verbose = show_messages

    print(f"Question: {question}\n")

    messages = [
        {"role": "system", "content": _DISPATCH_SYSTEM},
        {"role": "user",   "content": question}
    ]

    if show_messages:
        print("── OUTGOING MESSAGES ────────────────────────────────────")
        for m in messages:
            _print_message(m)

    # ── Round 1: ask GPT which tool(s) to call ───────────────────────────────
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )

    msg = response.choices[0].message

    if show_messages:
        print("\n── GPT TOOL SELECTION ───────────────────────────────────")
        _print_message(msg, label="ASSISTANT")

    # If GPT answered directly without using a tool, return that
    if not msg.tool_calls:
        print(f"\nAnswer: {msg.content}")
        return msg.content

    # ── Execute each tool and collect results ────────────────────────────────
    results = []
    for tool_call in msg.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        print(f"  → Calling {fn_name}({fn_args})")
        result = str(FUNCTION_MAP[fn_name](fn_args))
        results.append(result)
        if show_messages:
            print(f"\n── TOOL RESULT: {fn_name} ───────────────────────────────")
            print(f"  {result}")

    # ── Round 2: merge results into a single reply ───────────────────────────
    if len(results) == 1:
        answer = results[0]
    else:
        combined = "\n\n".join(f"Result {i+1}: {r}" for i, r in enumerate(results))
        if show_messages:
            print("\n── ROUND 2: merging results ─────────────────────────────")
        merge_resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "system", "content": _COMBINE_SYSTEM},
                {"role": "user",   "content": f"Question: {question}\n\n{combined}"},
            ],
        )
        answer = merge_resp.choices[0].message.content.strip()
        if show_messages:
            print(f"  Merged answer: {answer}")

    print(f"\nAnswer: {answer}")
    return answer

print("Dispatcher ready. Set show_messages=True in cell 5 to trace the API calls.")


Dispatcher ready. Set show_messages=True in cell 5 to trace the API calls.


In [49]:
## Cell 5 — Ask a question  ← edit QUESTION and SHOW_MESSAGES here

from IPython.display import display, HTML

QUESTION      = "What are the main problems Arsenal are facing?"
QUESTION      = "Is there alot written about Manchester City?"

SHOW_MESSAGES = True   # set to True to see every message sent to / received from the API

answer = dispatch(QUESTION, show_messages=SHOW_MESSAGES)

display(HTML(f"""
<div style="font-family:Georgia,serif; max-width:680px; margin-top:16px;
            border-left:4px solid #2a7ae2; padding:12px 20px;
            background:#f8f9fc; border-radius:4px; line-height:1.7;">
  <div style="font-size:0.8em; color:#888; margin-bottom:6px;">
    ⚽ {QUESTION}
  </div>
  <div style="font-size:1.05em; color:#1a1a1a;">{answer}</div>
</div>
"""))


Question: Is there alot written about Manchester City?

── OUTGOING MESSAGES ────────────────────────────────────

────────────────────────────────────────────────────────────
[SYSTEM]
  You are a football news assistant with access to lookup tools. Rules for tool
  selection: 1. If the user asks a specific question about a TEAM (e.g. 'What
  problems is Arsenal facing?', 'Did Chelsea win?'), call BOTH
  tell_me_about_team AND answer_team_question. 2. If the user asks a specific
  question about a PLAYER (e.g. 'Did Haaland score?', 'Is Salah injured?'), call
  BOTH tell_me_about_player AND answer_player_question. 3. If the user just asks
  for a summary or mentions a name only, call tell_me_about_team or
  tell_me_about_player. 4. If the user asks how often someone appears in the
  news, call how_many_articles. Always call tools — never answer from your own
  knowledge.

────────────────────────────────────────────────────────────
[USER]
  Is there alot written about Manchester City?

